# Day 4 · Lab 1 — LangSmith + OpenTelemetry Unified Observability

## What you'll build

1. A simple LangGraph agent (bureau lookup — reused from Day 1)
2. LangSmith `@traceable` decorator for cross-framework tracing
3. OpenTelemetry spans with **console exporter** (no Azure key needed)
4. Watch the SAME operation appear in both LangSmith UI and stdout
5. Understand the coexistence pattern for enterprise deployment

## Prerequisites

- Days 1–3 completed
- Same sandbox: `~/agentic-lab/.env` with `OPENROUTER_API_KEY`
- Same kernel: `/opt/miniconda/bin/python` (base)
- OTel packages (Step 1 auto-installs)

## Step 1 — Environment + package install

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Auto-install anything missing
def ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg.replace("-", "_").split("[")[0])
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

ensure("python-dotenv", "dotenv")
ensure("opentelemetry-api", "opentelemetry")
ensure("opentelemetry-sdk", "opentelemetry.sdk")

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "":
        del os.environ[k]

# Set OpenAI-compatible aliases
if os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

print(f"✓ OPENROUTER_API_KEY set: {bool(os.environ.get('OPENROUTER_API_KEY'))}")
langsmith_on = os.environ.get("LANGSMITH_TRACING","").lower() == "true" and os.environ.get("LANGSMITH_API_KEY")
print(f"✓ LangSmith tracing: {'ON' if langsmith_on else 'off (no key — still runs, just no LangSmith traces)'}")

## Step 2 — Set up OpenTelemetry with console exporter

For the lab we export spans to stdout. In production, swap `ConsoleSpanExporter` for `AzureMonitorTraceExporter` (or Datadog, or any OTLP endpoint). Same code, different exporter.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter

# Only configure once per process (Jupyter re-runs would double up otherwise)
if not isinstance(trace.get_tracer_provider(), TracerProvider):
    provider = TracerProvider()
    provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
    trace.set_tracer_provider(provider)

tracer = trace.get_tracer("day4_lab")
print("✓ OTel tracer configured with console exporter")

## Step 3 — LangSmith @traceable decorator

`@traceable` works even without a LangSmith key — it just no-ops. With a key, it sends traces to smith.langchain.com.

In [ ]:
try:
    from langsmith import traceable
    LANGSMITH_AVAILABLE = True
except ImportError:
    # Fallback: no-op decorator
    def traceable(fn=None, **kwargs):
        if fn is None:
            return lambda f: f
        return fn
    LANGSMITH_AVAILABLE = False

print(f"✓ LangSmith @traceable available: {LANGSMITH_AVAILABLE}")

## Step 4 — Instrument a function with BOTH LangSmith and OTel

This is the coexistence pattern. Same function, two observability layers.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


@traceable(name="bureau_lookup")   # LangSmith
def bureau_lookup(applicant_id: str, income: float) -> dict:
    # OTel span — nested inside the traceable
    with tracer.start_as_current_span("bureau.lookup") as span:
        span.set_attribute("applicant_id", applicant_id)
        span.set_attribute("income", income)
        
        prompt = f"Return JSON with 'score' (300-850) and 'tier' for income {income:.0f}. ONLY JSON."
        text = llm.invoke(prompt).content.strip()
        
        # Strip markdown fences if present
        if text.startswith("```"):
            text = text.strip("`").split("\n", 1)[-1].rsplit("```", 1)[0].strip()
            if text.startswith("json"):
                text = text[4:].strip()
        
        import json
        try:
            data = json.loads(text)
            score = int(data.get("score", 700))
        except Exception:
            score = 700
        
        span.set_attribute("bureau.score", score)
        return {"applicant_id": applicant_id, "score": score}


print("✓ bureau_lookup instrumented with both LangSmith and OTel")

## Step 5 — Run it and observe traces in stdout

Watch for the OTel span output right after the return value. In production, that JSON goes to Azure Monitor instead of stdout.

In [ ]:
result = bureau_lookup("APP-DAY4-001", 6000)
print(f"\nResult: {result}")

# Force flush so spans print now, not on process exit
trace.get_tracer_provider().force_flush()
print("\n✓ OTel spans flushed above (scroll up to see the JSON span output)")

## Step 6 — Multi-hop tracing: parent + child spans

Real production: one root span with multiple child spans for each downstream call. Watch the trace_id stay consistent.

In [ ]:
@traceable(name="loan_application")
def process_application(applicant_id: str, income: float) -> dict:
    with tracer.start_as_current_span("loan.process") as root:
        root.set_attribute("applicant_id", applicant_id)
        
        # Child 1: eligibility (deterministic)
        with tracer.start_as_current_span("eligibility.check") as span:
            score = min(100.0, (income * 12 / 200_000) * 25)
            passed = score >= 50
            span.set_attribute("eligibility.score", score)
            span.set_attribute("eligibility.passed", passed)
        
        if not passed:
            root.set_attribute("decision", "rejected_early")
            return {"decision": "rejected", "stage": "eligibility"}
        
        # Child 2: bureau (LLM call — has its own nested span)
        bureau = bureau_lookup(applicant_id, income)
        
        # Decision
        decision = "approved" if bureau["score"] >= 700 else "review"
        root.set_attribute("decision", decision)
        return {"decision": decision, "bureau": bureau}


result = process_application("APP-DAY4-MULTI", 6000)
print(f"\nResult: {result}")

trace.get_tracer_provider().force_flush()
print("\n✓ Multi-span trace flushed. Notice trace_id consistency across spans.")

## Step 7 — Search span output for correlation

The JSON above has `trace_id` and `span_id`. All spans in one application share a trace_id. Use this in Azure Monitor / KQL to view an entire workflow.

In [ ]:
# In production KQL / Log Analytics you'd run:
kql = '''
traces
| where operation_Id == "<trace_id_from_above>"
| project timestamp, name, duration_ms=toint(duration/1e6), attributes
| order by timestamp asc
'''
print("Example KQL query for Azure Monitor:")
print(kql)
print("\nEach span above has an operation_Id (trace_id) — that's your correlation key.")

## What you learned

1. **OTel setup** is 4 lines: provider, processor, exporter, get_tracer
2. **`@traceable`** gives you LangSmith coverage on any Python function
3. **`start_as_current_span`** gives you OTel coverage on any code block
4. **Both together** = LangSmith for dev UI + OTel for enterprise APM
5. **Parent-child spans** correlate via shared trace_id — critical for multi-service debugging
6. **Console exporter → Azure Monitor**: same code, one-line exporter swap

## Production notes

- Swap `ConsoleSpanExporter` → `AzureMonitorTraceExporter` with connection string
- Add `azure-monitor-opentelemetry-exporter` to your requirements
- Enable auto-instrumentation for httpx, psycopg — no code changes needed
- Sample at 1-10% in the collector to control cost

## Next

Open `lab2_injection_defense.ipynb` for prompt injection defense.